In [1]:
!pip install -q bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00


In [4]:
import json
import re
from pathlib import Path

# Change this later when you tell path
DATA_DIR = Path("/content/drive/MyDrive/data_v2")

channels = [
    "delhifoodwalks",
    "main_bhi_bharat",
    "masterchefnambie",
    "northeastindiafood",
    "roohi_haflongbar"
]

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# ✅ Quantization config (CORRECT WAY)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [5]:
print(model.device)

cuda:0


In [6]:
def is_non_empty(value):
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip() != ""
    if isinstance(value, list):
        return len(value) > 0
    return False


def get_category(description, transcription_en):
    if is_non_empty(description):
        return 0
    elif is_non_empty(transcription_en):
        return 1
    else:
        return 2

In [7]:
def remove_timestamps(lines):
    cleaned = []
    for line in lines:
        line = re.sub(r"\[\d{2}:\d{2}:\d{2}\.\d+\]", "", line)
        cleaned.append(line.strip())
    return cleaned


def smart_slice(text, max_chars=4500):
    n = len(text)
    if n <= max_chars:
        return text

    part = max_chars // 3
    return text[:part] + text[n//2 - part//2 : n//2 + part//2] + text[-part:]


FOOD_HINTS = ["ingredients", "cook", "recipe", "fry", "boil", "taste"]
NEWS_HINTS = ["news", "report", "update", "breaking", "government"]

def extract_signal_lines(lines, max_lines=10):
    selected = []
    for line in lines:
        l = line.lower()
        if any(k in l for k in FOOD_HINTS + NEWS_HINTS):
            selected.append(line)
        if len(selected) >= max_lines:
            break
    return " ".join(selected)

In [8]:
def build_cat0_input(title, description):
    return f"""
[TYPE: DESCRIPTION]

[TITLE]
{title}

[DESCRIPTION]
{description}
"""


def build_cat1_input(title, transcript_lines):
    cleaned = remove_timestamps(transcript_lines)
    joined = " ".join(cleaned)

    sliced = smart_slice(joined)
    signals = extract_signal_lines(cleaned)

    return f"""
[TYPE: TRANSCRIPT]

[TITLE]
{title}

[TRANSCRIPT_SNIPPET]
{sliced}

[IMPORTANT_LINES]
{signals}
"""


def build_cat2_input(title):
    return f"""
[TYPE: TITLE_ONLY]

[TITLE]
{title}
"""

In [9]:
def build_prompt(text):
    return f"""
You are a strict classifier.

Classify the YouTube video into ONE of the following categories:
- food
- news
- other
- unpredictable

Definitions:
- food: cooking, recipes, ingredients, food preparation, food reviews
- news: reporting events, updates, journalism, factual reporting
- other: travel, vlog, lifestyle, entertainment not focused on food or news
- unpredictable: not enough information to decide

Rules:
- Output ONLY one word
- No explanation

{text}

Answer:
"""

In [13]:
VALID = ["food", "news", "other", "unpredictable"]

def run_llm(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        temperature=0.0
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded


def parse_output(output):
    output = output.lower().strip()

    VALID = ["food", "news", "other", "unpredictable"]

    # scan from bottom (most reliable)
    for line in reversed(output.split("\n")):
        line = line.strip()

        for v in VALID:
            if line == v or line.endswith(v):
                return v

    return "unpredictable"

In [14]:
samples = {0: None, 1: None, 2: None}

for channel in channels:
    for file_path in (DATA_DIR / channel).glob("*.json"):

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        title = data.get("metadata", {}).get("title", "")
        description = data.get("metadata", {}).get("description", "")
        transcription_en = data.get("transcription_english", [])

        cat = get_category(description, transcription_en)

        if samples[cat] is None:
            samples[cat] = (file_path, title, description, transcription_en)

        if all(v is not None for v in samples.values()):
            break
    if all(v is not None for v in samples.values()):
        break

In [15]:
for cat in [0, 1, 2]:
    print("\n" + "="*80)
    print(f"🔍 CATEGORY {cat}")
    print("="*80)

    file_path, title, description, transcription_en = samples[cat]

    print(f"\n📁 FILE: {file_path}")
    print(f"\n📝 TITLE:\n{title}")

    if cat == 0:
        print(f"\n📄 DESCRIPTION (first 300 chars):\n{description[:300]}")
        text = build_cat0_input(title, description)

    elif cat == 1:
        print(f"\n📜 TRANSCRIPT SAMPLE (first 3 lines):")
        for l in transcription_en[:3]:
            print(l)

        text = build_cat1_input(title, transcription_en)

    else:
        text = build_cat2_input(title)

    print("\n" + "-"*60)
    print("📦 FINAL INPUT TO LLM:")
    print("-"*60)
    print(text[:1000])  # truncate for display

    prompt = build_prompt(text)

    print("\n" + "-"*60)
    print("🧠 PROMPT:")
    print("-"*60)
    print(prompt[:1200])

    output = run_llm(prompt)

    print("\n" + "-"*60)
    print("🤖 RAW MODEL OUTPUT:")
    print("-"*60)
    print(output)

    label = parse_output(output)

    print("\n" + "-"*60)
    print("✅ PARSED LABEL:")
    print("-"*60)
    print(label)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



🔍 CATEGORY 0

📁 FILE: /content/drive/MyDrive/Internship/NPTEL/data_v2/delhifoodwalks/110. BEST EVER Penang Street Food Tour.json

📝 TITLE:
BEST EVER Penang Street Food Tour in Malaysia 🇲🇾 I Nasi Kandar + Loh Bak + Char Hor Fun + Ais Kacang

📄 DESCRIPTION (first 300 chars):
Penang's culinary reflects its diverse cultural heritage. Guided by the knowledgeable Wei Shen Ooi from Simply Enak, the Penang street food tour was an immersive journey through the heart of George Town, where we explored a mix of traditional Malay, Chinese, and Indian dishes, as well as unique fusi

------------------------------------------------------------
📦 FINAL INPUT TO LLM:
------------------------------------------------------------

[TYPE: DESCRIPTION]

[TITLE]
BEST EVER Penang Street Food Tour in Malaysia 🇲🇾 I Nasi Kandar + Loh Bak + Char Hor Fun + Ais Kacang

[DESCRIPTION]
Penang's culinary reflects its diverse cultural heritage. Guided by the knowledgeable Wei Shen Ooi from Simply Enak, the Penang stree

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



------------------------------------------------------------
🤖 RAW MODEL OUTPUT:
------------------------------------------------------------

You are a strict classifier.

Classify the YouTube video into ONE of the following categories:
- food
- news
- other
- unpredictable

Definitions:
- food: cooking, recipes, ingredients, food preparation, food reviews
- news: reporting events, updates, journalism, factual reporting
- other: travel, vlog, lifestyle, entertainment not focused on food or news
- unpredictable: not enough information to decide

Rules:
- Output ONLY one word
- No explanation


[TYPE: DESCRIPTION]

[TITLE]
BEST EVER Penang Street Food Tour in Malaysia 🇲🇾 I Nasi Kandar + Loh Bak + Char Hor Fun + Ais Kacang

[DESCRIPTION]
Penang's culinary reflects its diverse cultural heritage. Guided by the knowledgeable Wei Shen Ooi from Simply Enak, the Penang street food tour was an immersive journey through the heart of George Town, where we explored a mix of traditional Malay, Chi

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



------------------------------------------------------------
🤖 RAW MODEL OUTPUT:
------------------------------------------------------------

You are a strict classifier.

Classify the YouTube video into ONE of the following categories:
- food
- news
- other
- unpredictable

Definitions:
- food: cooking, recipes, ingredients, food preparation, food reviews
- news: reporting events, updates, journalism, factual reporting
- other: travel, vlog, lifestyle, entertainment not focused on food or news
- unpredictable: not enough information to decide

Rules:
- Output ONLY one word
- No explanation


[TYPE: TRANSCRIPT]

[TITLE]
Unseen Wild Greens & Pitha of Odisha | Best Forest Breakfast at Simlipal 🌿l Dahi Moori chhena Kela

[TRANSCRIPT_SNIPPET]
Nothing more than that is nutritious. Nothing more nutritious than this.No and delicious too. Mustard puppy is a paste of seeds and jelly. Mustard puppy is a paste of seeds and a paste of jelly. All the local items of the Patwa are local. All the lo